In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer #makes sentnces into numerical embeddings (vectors) -> encoder
import faiss #used for similarity (like cosine similarity so it choses the most related text)

c:\Users\korej\anaconda3\envs\ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1- bring the data to be used
text = [
 
    # =========================
    # Customer Support (1–10)
    # =========================
    "How do I reset my password?",
    "Where can I update my email address?",
    "How do I cancel my subscription?",
    "What are your business hours?",
    "How can I change my billing information?",
    "How do I contact customer support?",
    "Where can I download my invoice?",
    "How do I track my order?",
    "What payment methods do you accept?",
    "How do I create a new account?",
 
    # =========================
    # Artificial Intelligence (11–20)
    # =========================
    "Machine learning is a branch of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "Natural language processing enables computers to understand text.",
    "Computer vision allows machines to analyze images.",
    "Generative AI can create text, images, and code.",
    "PyTorch is widely used for deep learning research.",
    "TensorFlow is a popular machine learning framework.",
    "Large language models are trained on massive text datasets.",
    "Data science combines statistics and programming.",
    "Feature engineering improves machine learning models.",
 
    # =========================
    # Programming (21–30)
    # =========================
    "Python is one of the most popular programming languages.",
    "Java is commonly used for enterprise applications.",
    "Git helps developers manage source code versions.",
    "GitHub is used to host software repositories.",
    "APIs allow different software systems to communicate.",
    "Object-oriented programming organizes code using classes.",
    "Debugging helps identify and fix software bugs.",
    "SQL is used to manage relational databases.",
    "Flutter enables cross-platform mobile app development.",
    "Docker simplifies application deployment using containers.",
 
    # =========================
    # Science & Education (31–40)
    # =========================
    "The Earth revolves around the Sun.",
    "Water boils at one hundred degrees Celsius.",
    "Gravity keeps planets in orbit around the Sun.",
    "Photosynthesis allows plants to produce food.",
    "Reading books improves vocabulary and knowledge.",
    "Mathematics is essential for engineering and science.",
    "Regular practice improves problem-solving skills.",
    "Online learning provides flexible education opportunities.",
    "The human brain contains billions of neurons.",
    "Electricity powers most modern electronic devices.",
 
    # =========================
    # Health, Travel & Lifestyle (41–50)
    # =========================
    "Regular exercise improves physical and mental health.",
    "A healthy diet includes fruits and vegetables.",
    "Drinking enough water keeps the body hydrated.",
    "Sleep is important for maintaining good health.",
    "Traveling helps people explore different cultures.",
    "Airplanes are one of the fastest modes of transportation.",
    "Hotels provide accommodation for travelers.",
    "Football is the world's most popular sport.",
    "Cricket is widely played in Pakistan and India.",
    "Cybersecurity protects systems from digital attacks."
 
]

In [3]:
# upload the pre-trained model mentioned
model = SentenceTransformer('all-MiniLM-L6-v2') #don't use sentnce-transformer by mistake, it's the package name not object or class

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1850.60it/s]


In [4]:
# 2- convert them into emeddings using faiss, but first make it float so faiss can work on it
embedding = model.encode(text, convert_to_numpy=True).astype('float32') # needs float32 not any float, the default is float64

In [5]:
matrix = embedding.shape[1]  # (rows 0, columns 1), so it relies on columns which is the traits to how much related it is
similarity = faiss.IndexFlatIP(matrix) #sees how much related the sentnces are
faiss.normalize_L2(embedding)  # shrinking all their lengths to 1, so it relies completely on how much related they are (cosine sim = dot/mag where mag=1)
similarity.add(embedding) # now we relate the sentnces cuz it's normalized too

In [6]:
# 3- we run the search since it's related now
query = ['when should i use AI?',
         'i forgot my email address',
         'which programming language is widely used nowadays?',
         'how do plants eat and grow?',
         'does travelin help me in any way?']

In [7]:
# 4- how the search will show up
k = 2 #the closest two points using KNN
for q in query:
        # we should encode, normalize the query too
        qEmbed = model.encode([q], convert_to_numpy=True).astype('float32') 
        faiss.normalize_L2(qEmbed)
        distance, answer = similarity.search(qEmbed, k) # answer will be used to give the index of the top 2 related answers, while distance will give range 0-1 of how much close they are, .search gives dot product and te closest two answers
        print(f'Question: {q}')
        print(f'top 2 matches:')
        for i, idx in enumerate(answer[0]):   #answer= [i number of the loop, idx number of the sentnce], distance =[0] so it can be replaced by the cosine score
                print(f'{i+1}.{text[idx]}\n similarity score: {distance[0][i]:.2f} \n') #i+1 so it doesn't start from 0
        print('-----------------------------\n')

Question: when should i use AI?
top 2 matches:
1.Generative AI can create text, images, and code.
 similarity score: 0.47 

2.Machine learning is a branch of artificial intelligence.
 similarity score: 0.44 

-----------------------------

Question: i forgot my email address
top 2 matches:
1.Where can I update my email address?
 similarity score: 0.62 

2.How do I reset my password?
 similarity score: 0.47 

-----------------------------

Question: which programming language is widely used nowadays?
top 2 matches:
1.Python is one of the most popular programming languages.
 similarity score: 0.68 

2.Java is commonly used for enterprise applications.
 similarity score: 0.55 

-----------------------------

Question: how do plants eat and grow?
top 2 matches:
1.Photosynthesis allows plants to produce food.
 similarity score: 0.65 

2.A healthy diet includes fruits and vegetables.
 similarity score: 0.37 

-----------------------------

Question: does travelin help me in any way?
top 2 ma